In [1]:
import pandas as pd

In [2]:
model_df = pd.read_excel("model_df_clean.xlsx")

source = model_df[["Flight_ID", "Fleet", "Gross_Weight_At_Liftoff_kg"]].merge(
   model_df[["Flight_ID", "Great_Circle_Distance_NM"]],
    on="Flight_ID", how="left"
)

print(source.shape)
print(source.isna().sum())

(1404, 4)
Flight_ID                     0
Fleet                         0
Gross_Weight_At_Liftoff_kg    0
Great_Circle_Distance_NM      0
dtype: int64


In [3]:
model_df.columns

Index(['Flight_ID', 'Flight_Number', 'Date', 'Fleet', 'Aircraft_Registration',
       'Route', 'Total_Distance_Flown_NM', 'Great_Circle_Distance_NM',
       'Gross_Weight_At_Liftoff_kg', 'Max_Tailwind_During_Takeoff_kt',
       'Max_Static_Air_Temperature_degree_celsius', 'Total_Fuel_Burn_kg'],
      dtype='object')

In [4]:
print(source["Fleet"].value_counts())

Fleet
B737-NG     1074
A330         283
B737F-NG      47
Name: count, dtype: int64


In [5]:
print(source.duplicated(subset=["Flight_ID"]).sum())
print(model_df.duplicated(subset=["Flight_ID"]).sum())

0
0


In [6]:
source = source.drop_duplicates(subset=["Flight_ID"], keep="first")
print(source.shape)
print(source["Fleet"].value_counts())

(1404, 4)
Fleet
B737-NG     1074
A330         283
B737F-NG      47
Name: count, dtype: int64


In [7]:
import pandas as pd
import numpy as np
from scipy.stats import gaussian_kde

# ── Load and merge ───────────────────────────────────────────────────────
full_df = pd.read_excel("fuel_df_15_09_26.xlsx")
model_df = pd.read_excel("model_df_clean.xlsx")

source = model_df[["Flight_ID", "Fleet", "Gross_Weight_At_Liftoff_kg"]].merge(
    full_df[["Flight_ID", "Great_Circle_Distance_NM"]],
    on="Flight_ID", how="left"
)

# ── Remove duplicate Flight_IDs introduced by merging against the
#    pre-dedup full_df ───────────────────────────────────────────────────
source = source.drop_duplicates(subset=["Flight_ID"], keep="first")

# ── Strip units and convert distance to numeric ─────────────────────────
source["Great_Circle_Distance_NM"] = (
    source["Great_Circle_Distance_NM"]
    .astype(str)
    .str.extract(r"([-+]?\d*\.?\d+)")[0]
    .astype(float)
)

print("Source shape:", source.shape)
print(source["Fleet"].value_counts())
print(source["Great_Circle_Distance_NM"].describe())

# ── Fit one joint KDE per fleet, in log space ───────────────────────────
BANDWIDTH_MULTIPLIER = {"B737F-NG": 1.8}
DEFAULT_MULTIPLIER = 1.0

kde_models = {}

for fleet, grp in source.groupby("Fleet"):
    log_dist = np.log(grp["Great_Circle_Distance_NM"].values)
    log_wt = np.log(grp["Gross_Weight_At_Liftoff_kg"].values)
    data = np.vstack([log_dist, log_wt])

    kde_auto = gaussian_kde(data)
    multiplier = BANDWIDTH_MULTIPLIER.get(fleet, DEFAULT_MULTIPLIER)
    bw = kde_auto.factor * multiplier
    kde = gaussian_kde(data, bw_method=bw)

    kde_models[fleet] = {
        "kde": kde,
        "n": len(grp),
        "dist_min": grp["Great_Circle_Distance_NM"].min(),
        "dist_max": grp["Great_Circle_Distance_NM"].max(),
        "weight_min": grp["Gross_Weight_At_Liftoff_kg"].min(),
        "weight_max": grp["Gross_Weight_At_Liftoff_kg"].max(),
    }

    print(f"\n{fleet}: n={len(grp)}, bandwidth={bw:.4f} (x{multiplier})")
    print(f"  distance range: {kde_models[fleet]['dist_min']:.0f} - {kde_models[fleet]['dist_max']:.0f} NM")
    print(f"  weight range:   {kde_models[fleet]['weight_min']:.0f} - {kde_models[fleet]['weight_max']:.0f} kg")

Source shape: (1404, 4)
Fleet
B737-NG     1074
A330         283
B737F-NG      47
Name: count, dtype: int64
count    1404.000000
mean     1071.724145
std      1043.330903
min        39.200000
25%       213.800000
50%       601.100000
75%      1455.000000
max      3567.000000
Name: Great_Circle_Distance_NM, dtype: float64

A330: n=283, bandwidth=0.3903 (x1.0)
  distance range: 136 - 3567 NM
  weight range:   137931 - 228376 kg

B737-NG: n=1074, bandwidth=0.3125 (x1.0)
  distance range: 39 - 2196 NM
  weight range:   48782 - 76961 kg

B737F-NG: n=47, bandwidth=0.9475 (x1.8)
  distance range: 183 - 2205 NM
  weight range:   47392 - 76404 kg


In [8]:
def generate_distance_weight(fleet, max_attempts=50, rng=None):
    f = kde_models[fleet]
    kde = f["kde"]
    rng = rng or np.random.default_rng()

    for _ in range(max_attempts):
        sample = kde.resample(1, seed=rng)
        distance = float(np.exp(sample[0, 0]))
        weight = float(np.exp(sample[1, 0]))

        if (f["dist_min"] <= distance <= f["dist_max"] and
                f["weight_min"] <= weight <= f["weight_max"]):
            return distance, weight

    # Fallback: clip rather than loop forever on a bad draw
    distance = min(max(distance, f["dist_min"]), f["dist_max"])
    weight = min(max(weight, f["weight_min"]), f["weight_max"])
    return distance, weight


rng = np.random.default_rng(0)
for fleet in kde_models:
    print(f"\n{fleet}:")
    for i in range(5):
        d, w = generate_distance_weight(fleet, rng=rng)
        print(f"  draw {i+1}: distance={d:>8,.0f} NM   weight={w:>9,.0f} kg")


A330:
  draw 1: distance=   3,364 NM   weight=  223,352 kg
  draw 2: distance=   3,221 NM   weight=  180,515 kg
  draw 3: distance=     404 NM   weight=  172,035 kg
  draw 4: distance=     305 NM   weight=  160,342 kg
  draw 5: distance=   2,781 NM   weight=  220,691 kg

B737-NG:
  draw 1: distance=     123 NM   weight=   53,183 kg
  draw 2: distance=     141 NM   weight=   55,283 kg
  draw 3: distance=     124 NM   weight=   54,876 kg
  draw 4: distance=     246 NM   weight=   60,858 kg
  draw 5: distance=     350 NM   weight=   54,330 kg

B737F-NG:
  draw 1: distance=     673 NM   weight=   70,288 kg
  draw 2: distance=     786 NM   weight=   70,184 kg
  draw 3: distance=     552 NM   weight=   70,512 kg
  draw 4: distance=     673 NM   weight=   59,917 kg
  draw 5: distance=     594 NM   weight=   74,040 kg


In [9]:
rng = np.random.default_rng(1)
draws = [generate_distance_weight("B737F-NG", rng=rng) for _ in range(30)]
distances = [d for d, w in draws]
weights = [w for d, w in draws]

print(f"Distance: min={min(distances):.0f}, max={max(distances):.0f}, "
      f"mean={sum(distances)/len(distances):.0f}")
print(f"Weight:   min={min(weights):.0f}, max={max(weights):.0f}, "
      f"mean={sum(weights)/len(weights):.0f}")

Distance: min=219, max=2041, mean=966
Weight:   min=49015, max=75661, mean=64234


In [10]:
source_3d = model_df[["Flight_ID", "Fleet", "Total_Distance_Flown_NM",
                       "Gross_Weight_At_Liftoff_kg",
                       "Max_Tailwind_During_Takeoff_kt"]].dropna()

print(source_3d.shape)
print(source_3d["Fleet"].value_counts())

(1404, 5)
Fleet
B737-NG     1074
A330         283
B737F-NG      47
Name: count, dtype: int64


In [11]:
kde_models_3d = {}

for fleet, grp in source_3d.groupby("Fleet"):
    log_dist = np.log(grp["Total_Distance_Flown_NM"].values)
    log_wt = np.log(grp["Gross_Weight_At_Liftoff_kg"].values)
    tailwind = grp["Max_Tailwind_During_Takeoff_kt"].values

    data = np.vstack([log_dist, log_wt, tailwind])

    kde_auto = gaussian_kde(data)
    multiplier = BANDWIDTH_MULTIPLIER.get(fleet, 1.0)
    bw = kde_auto.factor * multiplier
    kde = gaussian_kde(data, bw_method=bw)

    kde_models_3d[fleet] = {
        "kde": kde,
        "n": len(grp),
        "dist_min": grp["Total_Distance_Flown_NM"].min(),
        "dist_max": grp["Total_Distance_Flown_NM"].max(),
        "weight_min": grp["Gross_Weight_At_Liftoff_kg"].min(),
        "weight_max": grp["Gross_Weight_At_Liftoff_kg"].max(),
        "tailwind_min": grp["Max_Tailwind_During_Takeoff_kt"].min(),
        "tailwind_max": grp["Max_Tailwind_During_Takeoff_kt"].max(),
    }

    print(f"{fleet}: n={len(grp)}, bandwidth={bw:.4f}")
    print(f"  distance:  {kde_models_3d[fleet]['dist_min']:.0f} - {kde_models_3d[fleet]['dist_max']:.0f} NM")
    print(f"  weight:    {kde_models_3d[fleet]['weight_min']:.0f} - {kde_models_3d[fleet]['weight_max']:.0f} kg")
    print(f"  tailwind:  {kde_models_3d[fleet]['tailwind_min']:.0f} - {kde_models_3d[fleet]['tailwind_max']:.0f} kt")

A330: n=283, bandwidth=0.4464
  distance:  145 - 4261 NM
  weight:    137931 - 228376 kg
  tailwind:  -14 - 10 kt
B737-NG: n=1074, bandwidth=0.3690
  distance:  44 - 2589 NM
  weight:    48782 - 76961 kg
  tailwind:  -16 - 12 kt
B737F-NG: n=47, bandwidth=1.0385
  distance:  208 - 2574 NM
  weight:    47392 - 76404 kg
  tailwind:  -11 - 5 kt


In [15]:
def generate_flight_inputs(fleet, max_attempts=50, rng=None):
    f = kde_models_3d[fleet]
    kde = f["kde"]
    rng = rng or np.random.default_rng()

    for _ in range(max_attempts):
        sample = kde.resample(1, seed=rng)
        distance = float(np.exp(sample[0, 0]))
        weight = float(np.exp(sample[1, 0]))
        tailwind = float(sample[2, 0])

        if (f["dist_min"] <= distance <= f["dist_max"] and
                f["weight_min"] <= weight <= f["weight_max"] and
                f["tailwind_min"] <= tailwind <= f["tailwind_max"]):
            return distance, weight, tailwind

    # Fallback: clip rather than loop forever
    distance = min(max(distance, f["dist_min"]), f["dist_max"])
    weight = min(max(weight, f["weight_min"]), f["weight_max"])
    tailwind = min(max(tailwind, f["tailwind_min"]), f["tailwind_max"])
    return distance, weight, tailwind


rng = np.random.default_rng(0)
for fleet in kde_models_3d:
    print(f"\n{fleet}:")
    for i in range(5):
        d, w, t = generate_flight_inputs(fleet, rng=rng)
        print(f"  draw {i+1}: distance={d:>8,.0f} NM   weight={w:>9,.0f} kg   tailwind={t:>6.1f} kt")


A330:
  draw 1: distance=   4,001 NM   weight=  212,876 kg   tailwind=  -1.3 kt
  draw 2: distance=     165 NM   weight=  161,706 kg   tailwind=  -1.9 kt
  draw 3: distance=     261 NM   weight=  156,621 kg   tailwind=  -1.2 kt
  draw 4: distance=     800 NM   weight=  160,208 kg   tailwind=   3.3 kt
  draw 5: distance=   3,766 NM   weight=  183,870 kg   tailwind=  -2.0 kt

B737-NG:
  draw 1: distance=     364 NM   weight=   54,196 kg   tailwind=   3.3 kt
  draw 2: distance=     287 NM   weight=   62,954 kg   tailwind=  -3.0 kt
  draw 3: distance=     158 NM   weight=   56,875 kg   tailwind=   1.0 kt
  draw 4: distance=   1,659 NM   weight=   65,672 kg   tailwind=   0.6 kt
  draw 5: distance=     381 NM   weight=   57,025 kg   tailwind=  -2.8 kt

B737F-NG:
  draw 1: distance=     626 NM   weight=   58,587 kg   tailwind=  -1.0 kt
  draw 2: distance=     682 NM   weight=   69,431 kg   tailwind=  -5.7 kt
  draw 3: distance=     834 NM   weight=   67,928 kg   tailwind=   0.1 kt
  draw 4: 

In [16]:
import dill
with open("kde_models_3d.pkl", "wb") as f:
    dill.dump(kde_models_3d, f)
print("Saved with dill")

Saved with dill


In [17]:
print(list(kde_models_3d.keys()))

['A330', 'B737-NG', 'B737F-NG']
